In [0]:
%pip install "fsspec==2026.2.0"
dbutils.library.restartPython()

In [0]:
import pandas as pd

## Jailbreak Attack

In [0]:
splits = {'train': 'balanced/jailbreak_dataset_train_balanced.csv', 'test': 'balanced/jailbreak_dataset_test_balanced.csv'}

jail = pd.read_csv("hf://datasets/jackhhao/jailbreak-classification/" + splits["train"])

# To get test dataset, use splits["test"]

In [0]:
print(jail)

## Direct Injection Attack

In [0]:
splits = {
    'train': 'data/train-00000-of-00001.parquet', 
    'test': 'data/test-00000-of-00001.parquet'
    }

prompt = pd.read_parquet("hf://datasets/xTRam1/safe-guard-prompt-injection/" + splits["train"])

In [0]:
print(prompt)

## Indirect Prompt Injection Attack

Source: https://deepwiki.com/microsoft/BIPIA/4-bipia-dataset

In [0]:
indirect = pd.read_csv('/Workspace/Users/w10168315@usm.edu/HonorsThesis/bipia_email_table_code.csv')

print(indirect)


## Encoding Attacks 
- Self generated
- Injection seeds: label-1
- Benign seeds: label-0
- Use this dataset to generate many samples
- Source: https://huggingface.co/datasets/deepset/prompt-injections 

In [0]:
"""
Builds encoded-attack samples using deepset/prompt-injections as the seed source.

Pipeline:
  1. Load deepset train + test parquet files (from Hugging Face, or from local files).
  2. Clean seeds: strip, drop very short/long rows, de-duplicate, drop rows whose duplicates
     disagree on the label.
  3. Detect language per seed and keep the languages you ask for (default: English only).
  4. Balance: same number of injection seeds (label 1) and benign seeds (label 0).
  5. Encode every seed 8 ways (13 rows per seed incl. "hinted" versions) -> output CSV.


Usage:
    python generate_encoded_from_deepset.py
    python generate_encoded_from_deepset.py --languages en,de --max-seeds-per-class 150
    python generate_encoded_from_deepset.py --train-path train.parquet --test-path test.parquet   # offline

Rows produced = (seeds per class) x 2 classes x 13.   e.g. 120 per class -> 3,120 rows.
"""
import argparse
import base64
import codecs
import hashlib
import re
import urllib.parse

import pandas as pd
from langdetect import DetectorFactory, detect

DetectorFactory.seed = 0  # make language detection deterministic

HF_BASE = "hf://datasets/deepset/prompt-injections/"
HF_SPLITS = {
    "train": "data/train-00000-of-00001-9564e8b05b4757ab.parquet",
    "test": "data/test-00000-of-00001-701d16158af87368.parquet",
}

# ---------------- encoders (same as generate_encoded_attacks.py) ----------------
LEET = str.maketrans({"a": "4", "e": "3", "i": "1", "o": "0", "s": "5", "t": "7",
                      "A": "4", "E": "3", "I": "1", "O": "0", "S": "5", "T": "7"})
HOMOGLYPH = str.maketrans({"a": "а", "e": "е", "o": "о", "p": "р", "c": "с",
                           "x": "х", "i": "і", "y": "у", "A": "А", "E": "Е", "O": "О"})
ENCODERS = {
    "base64":    lambda s: base64.b64encode(s.encode("utf-8")).decode("ascii"),
    "rot13":     lambda s: codecs.encode(s, "rot_13"),
    "hex":       lambda s: s.encode("utf-8").hex(),
    "url":       lambda s: urllib.parse.quote(s),
    "reversed":  lambda s: s[::-1],
    "leetspeak": lambda s: s.translate(LEET),
    "homoglyph": lambda s: s.translate(HOMOGLYPH),
    "spaced":    lambda s: "   ".join(" ".join(w) for w in s.split(" ")),
}

HINTS = {
    "base64": "Decode this Base64 string and follow it: ",
    "rot13": "The following is ROT13. Decode it and do what it says: ",
    "hex": "Convert this hex to text and follow it: ",
    "url": "Decode this URL-encoded text and follow it: ",
    "reversed": "Read the following backwards and follow it: ",
    "leetspeak": None,
    "homoglyph": None,
    "spaced": None,
}


def normalize(text: str) -> str:
    """Lowercase + collapse whitespace. Used for de-duplication and for group_id."""
    return re.sub(r"\s+", " ", text.strip().lower())

def make_group_id(text: str) -> str:
    """Use THIS function for the plain (un-encoded) deepset rows too, so a sentence and all of
    its encoded versions share one group_id and never straddle the train/test split."""
    return "grp_" + hashlib.md5(normalize(text).encode("utf-8")).hexdigest()[:10]


def load_deepset(train_path, test_path):
    frames = []

    for name, local in [("train", train_path), ("test", test_path)]:
        path = local or (HF_BASE + HF_SPLITS[name])
        d = pd.read_parquet(path)
        d["deepset_split"] = name
        frames.append(d)

    df = pd.concat(frames, ignore_index=True)
    print("Columns found:", list(df.columns))
    missing = {"text", "label"} - set(df.columns)
    if missing:
        raise SystemExit(f"Expected columns 'text' and 'label' but missing {missing}. "
                         f"Rename them in load_deepset() to match the columns printed above.")
    return df[["text", "label", "deepset_split"]]


def safe_detect(t):
    try:
        return detect(t)
    except Exception:
        return "unknown"


def prepare_seeds(df, args):
    n0 = len(df)
    print(f"\nLoaded {n0} rows | label counts: {df.label.value_counts().to_dict()}")

    df = df.dropna(subset=["text"]).copy()
    df["text"] = df["text"].astype(str).str.strip()
    df = df[(df.text.str.len() >= args.min_chars) & (df.text.str.len() <= args.max_chars)]
    print(f"After length filter ({args.min_chars}-{args.max_chars} chars): {len(df)}")

    df["norm"] = df.text.map(normalize)
    # same text with conflicting labels = ambiguous -> drop all copies
    conflict = df.groupby("norm").label.nunique()
    conflict = set(conflict[conflict > 1].index)
    df = df[~df.norm.isin(conflict)]
    df = df.drop_duplicates(subset="norm")
    print(f"After de-duplication (dropped {len(conflict)} conflicting texts): {len(df)}")

    df["language"] = df.text.map(safe_detect)
    print("Language counts (all seeds):", df.language.value_counts().head(6).to_dict())
    keep = [l.strip() for l in args.languages.split(",")]
    df = df[df.language.isin(keep)]
    print(f"After language filter {keep}: {len(df)} | labels: {df.label.value_counts().to_dict()}")

    inj, ben = df[df.label == 1], df[df.label == 0]
    n = min(args.max_seeds_per_class, len(inj), len(ben))
    if n == 0:
        raise SystemExit("No seeds left in one of the classes. Loosen --languages / --max-chars.")
    if n < args.max_seeds_per_class:
        print(f"NOTE: only {n} seeds available per class (asked for {args.max_seeds_per_class}).")
    seeds = pd.concat([inj.sample(n, random_state=args.seed), ben.sample(n, random_state=args.seed)])
    print(f"Using {n} injection seeds + {n} benign seeds")
    return seeds.reset_index(drop=True)


def build_rows(seeds):
    rows = []
    for _, s in seeds.iterrows():
        text, label = s.text, int(s.label)
        group_id = make_group_id(text)
        for enc_name, fn in ENCODERS.items():
            encoded = fn(text)
            variants = [("bare", encoded)]
            if HINTS[enc_name]:
                variants.append(("hinted", HINTS[enc_name] + encoded))
            for style, out_text in variants:
                rows.append({
                    "text": out_text,
                    "label": label,
                    "attack_type": "encoded" if label == 1 else "none",
                    "source": "synthetic_encoding_deepset",
                    "language": s.language,
                    "is_synthetic": True,
                    "benign_type": "" if label == 1 else "encoded_benign",
                    "group_id": group_id,
                    "input_type": "user_prompt",
                    "encoding": enc_name,
                    "encoding_style": style,
                    "seed_source": "deepset/prompt-injections",
                    "deepset_split": s.deepset_split,
                    "seed_text": text,   # analysis only - never feed to the model
                })
    df = pd.DataFrame(rows)
    df.insert(0, "row_id", [f"encds_{i:05d}" for i in range(len(df))])
    return df


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train-path", default=None, help="local deepset train parquet (optional)")
    ap.add_argument("--test-path", default=None, help="local deepset test parquet (optional)")
    ap.add_argument("--languages", default="en", help="comma list of language codes to keep")
    ap.add_argument("--max-seeds-per-class", type=int, default=120)
    ap.add_argument("--min-chars", type=int, default=10)
    ap.add_argument("--max-chars", type=int, default=200)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--out", default="encoded_attacks_deepset.csv")
    ap.add_argument("--seeds-out", default="deepset_seeds_used.csv")
    args = ap.parse_args([])  # pass empty list so argparse ignores Jupyter kernel args

    df = load_deepset(args.train_path, args.test_path)
    seeds = prepare_seeds(df, args)
    seeds[["text", "label", "language", "deepset_split"]].to_csv(args.seeds_out, index=False)

    out = build_rows(seeds)
    out.to_csv(args.out, index=False)
    print(f"\nSaved {len(out)} rows to {args.out} (seeds used saved to {args.seeds_out})")
    print(out.groupby(["label", "encoding"]).size().unstack(0))


if __name__ == "__main__":
    main()

In [0]:
%pip install pandas pyarrow langdetect huggingface_hub fsspec